# EvoHash GigaEvolve Run

Notebook entrypoint: configure one hash target, run checks, verify the resolved GigaEvolve command, launch evolution in the background, monitor generation progress, then export the best programs.

In [ ]:
from pathlib import Path
import importlib
import sys

def find_project_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / 'stage3_gigaevolve').exists():
            return candidate
    raise RuntimeError(f'Could not find project root from {cur}')

PROJECT_ROOT = find_project_root()
PACKAGE_PARENT = PROJECT_ROOT.parent
if str(PACKAGE_PARENT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_PARENT))

from IPython.display import HTML, display
from evohash.stage3_gigaevolve import notebook_api as evo
importlib.reload(evo)

print('PROJECT_ROOT =', PROJECT_ROOT)


## Config

In [ ]:
import os

# Debug preset: tiny run to validate GigaEvolve + LLM + evaluator plumbing.
DEBUG_SMOKE = False

# Target task
HASH_ID = 'photodna'                 # phash | pdq | neuralhash | photodna
DATASET = str(PROJECT_ROOT / 'data' / 'dataset_v1_public.zip')
MAX_PAIRS = 30
BUDGET = 10000
GENERATIONS = 100
RUN_NAME = f'evohash_GigaEvolve_{HASH_ID}_EVO2'

# Initial programs
SEED_CONFIG = 'results/attack_configs_optimized.json'
SEED_ATTACKS = 'all'              # all or comma-separated attack ids, e.g. atkscopes or nes,simba

# Evaluation parallelism inside each candidate evaluation
EVAL_BACKEND = 'auto'             # auto | process | thread | sequential
EVAL_WORKERS = None               # None = existing auto resource policy
WORKER_CAP = 30

# GigaEvolve defaults from config/constants/evolution.yaml
MAX_MUTATIONS_PER_GENERATION = 10
MAX_ELITES_PER_GENERATION = 5

# Timeout limits. stage_timeout is the hard cap for one candidate DAG stage
# such as CallProgramFunction. 20 minutes keeps bad mutations from blocking
# an evolution generation for the old 40 minute default.
STAGE_TIMEOUT = 20 * 60
DAG_TIMEOUT = 60 * 60

# LLM
LLM_SOURCE = 'local'         # openrouter | local
OPENROUTER_MODEL = 'gpt_oss_120b'
LOCAL_MODEL = 'qwen3.6-27b-fp8'
LOCAL_BASE_URL = 'http://localhost:8385/v1'
LLM_TEMPERATURE = 0.6
LLM_MAX_TOKENS = 25000
LLM_REQUEST_TIMEOUT = 600
LLM_MAX_RETRIES = 2
REQUIRE_PROXY = os.environ.get('EVOHASH_REQUIRE_PROXY', '0') == '1'

# Redis / outputs
REDIS_HOST = 'localhost'
REDIS_PORT = 6379
REDIS_DB = 0
RESUME_RUN = False
CLEAR_REDIS_PREFIX_BEFORE_RUN = True
TRACKING_ENABLED = False
PROBLEM_OUTPUT_DIR = 'outputs/gigaevolve_problems'
LOG_DIR = 'outputs/gigaevolve_run_logs'
EXPORT_DIR = 'outputs/gigaevolve_exports'
SNAPSHOT_DIR = 'outputs/gigaevolve_snapshots'

# Flow switches
START_REDIS = False
CHECK_LLM = True
REQUIRE_CHECKS_OK = True
RUN_EVOLUTION = True
RUN_IN_BACKGROUND = True
MONITOR_IN_THIS_NOTEBOOK = True
EXPORT_AFTER_RUN = True

MODEL = OPENROUTER_MODEL if LLM_SOURCE == 'openrouter' else LOCAL_MODEL
display({
    'debug_smoke': DEBUG_SMOKE,
    'run_name': RUN_NAME,
    'hash': HASH_ID,
    'llm_source': LLM_SOURCE,
    'model': MODEL,
    'base_url': LOCAL_BASE_URL if LLM_SOURCE == 'local' else 'OpenRouter',
    'generations': GENERATIONS,
    'pairs': MAX_PAIRS,
    'budget': BUDGET,
    'seed_attacks': SEED_ATTACKS,
    'mutations_per_generation': MAX_MUTATIONS_PER_GENERATION,
    'stage_timeout': STAGE_TIMEOUT,
    'dag_timeout': DAG_TIMEOUT,
    'require_proxy': REQUIRE_PROXY,
})


## Models

In [ ]:
display(evo.model_table())

## Checks

The checks cell is intentionally non-throwing, so notebook output stays readable. If a required check fails, later cells will skip the run.

In [ ]:
if START_REDIS:
    display(evo.start_redis())

preflight = evo.preflight(
    dataset=DATASET,
    llm_source=LLM_SOURCE,
    model=MODEL,
    local_base_url=LOCAL_BASE_URL,
    redis_host=REDIS_HOST,
    redis_port=REDIS_PORT,
    redis_db=REDIS_DB,
    check_redis=True,
    check_llm_connection=CHECK_LLM,
    diagnostic_llm=False,
)
checks = evo.preflight_table(preflight)
display(checks)

def _display_rows(rows):
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        display(rows)

def _status_counts_text(payload):
    counts = payload.get('status_counts') or {}
    return ', '.join(f'{k}={v}' for k, v in counts.items() if v) or 'no programs'

redis_rows = []
redis_lock = evo.redis_lock_status(prefix=RUN_NAME, host=REDIS_HOST, port=REDIS_PORT, db=REDIS_DB)
redis_rows.append({
    'check': 'redis lock',
    'ok': bool(redis_lock.get('ok') and not redis_lock.get('locked')),
    'details': 'free' if not redis_lock.get('locked') else str(redis_lock.get('value')),
    'action': '' if not redis_lock.get('locked') else 'clear stale lock only after checking no run is alive',
})

redis_prefix_before = evo.redis_prefix_status(prefix=RUN_NAME, host=REDIS_HOST, port=REDIS_PORT, db=REDIS_DB)
redis_rows.append({
    'check': 'redis prefix before',
    'ok': bool(redis_prefix_before.get('ok')),
    'details': f"{redis_prefix_before.get('key_count', 0)} keys; {_status_counts_text(redis_prefix_before)}",
    'action': 'will clear for fresh run' if CLEAR_REDIS_PREFIX_BEFORE_RUN and redis_prefix_before.get('has_data') and not RESUME_RUN else '',
})

clear_result = None
if CLEAR_REDIS_PREFIX_BEFORE_RUN and redis_prefix_before.get('has_data') and not RESUME_RUN:
    clear_result = evo.clear_redis_prefix(prefix=RUN_NAME, host=REDIS_HOST, port=REDIS_PORT, db=REDIS_DB)
    redis_rows.append({
        'check': 'redis prefix cleanup',
        'ok': bool(clear_result.get('ok')),
        'details': f"deleted {clear_result.get('deleted', 0)} / matched {clear_result.get('matched', 0)} keys",
        'action': '',
    })

redis_prefix = evo.redis_prefix_status(prefix=RUN_NAME, host=REDIS_HOST, port=REDIS_PORT, db=REDIS_DB)
redis_rows.append({
    'check': 'redis prefix ready',
    'ok': bool(not redis_prefix.get('has_data') or RESUME_RUN),
    'details': f"{redis_prefix.get('key_count', 0)} keys; {_status_counts_text(redis_prefix)}",
    'action': '' if (not redis_prefix.get('has_data') or RESUME_RUN) else 'set CLEAR_REDIS_PREFIX_BEFORE_RUN=True or RESUME_RUN=True',
})
_display_rows(redis_rows)

REDIS_PREFIX_READY = bool(not redis_prefix.get('has_data') or RESUME_RUN)
CHECKS_OK = bool(preflight.get('ok')) or not REQUIRE_CHECKS_OK
overall_ok = CHECKS_OK and REDIS_PREFIX_READY
color = '#16794c' if overall_ok else '#b42318'
label = 'OK' if overall_ok else 'BLOCKED'
display(HTML(f"<div style='font-weight:600;color:{color};padding:4px 0'>Checks: {label}</div>"))
if not CHECKS_OK:
    display(preflight)

## Dry Run

This generates the problem package and prints the exact GigaEvolve command without starting evolution.

In [ ]:
dry = evo.run_gigaevolve(
    hash_id=HASH_ID,
    dataset=DATASET,
    run_name=RUN_NAME,
    max_pairs=MAX_PAIRS,
    budget=BUDGET,
    generations=GENERATIONS,
    eval_backend=EVAL_BACKEND,
    eval_workers=EVAL_WORKERS,
    worker_cap=WORKER_CAP,
    redis_host=REDIS_HOST,
    redis_port=REDIS_PORT,
    redis_db=REDIS_DB,
    tracking=TRACKING_ENABLED,
    llm_source=LLM_SOURCE,
    model=MODEL,
    local_base_url=LOCAL_BASE_URL,
    max_mutations_per_generation=MAX_MUTATIONS_PER_GENERATION,
    max_elites_per_generation=MAX_ELITES_PER_GENERATION,
    stage_timeout=STAGE_TIMEOUT,
    dag_timeout=DAG_TIMEOUT,
    llm_temperature=LLM_TEMPERATURE,
    llm_max_retries=LLM_MAX_RETRIES,
    llm_max_tokens=LLM_MAX_TOKENS,
    llm_request_timeout=LLM_REQUEST_TIMEOUT,
    resume=RESUME_RUN,
    clear_existing_prefix=False,
    problem_output_dir=PROBLEM_OUTPUT_DIR,
    seed_config=SEED_CONFIG,
    seed_attacks=SEED_ATTACKS,
    require_proxy=REQUIRE_PROXY,
    dry_run=True,
    stream=False,
)
DRY_RUN_OK = bool(dry.get('ok'))
print('DRY_RUN_OK =', DRY_RUN_OK)
print((dry.get('stdout') or '')[-2500:])
if dry.get('stderr'):
    print((dry.get('stderr') or '')[-2500:])


## Launch

Background mode writes all GigaEvolve output to `run.log`; the notebook only shows structured status.

In [ ]:
CAN_RUN = RUN_EVOLUTION and CHECKS_OK and DRY_RUN_OK and REDIS_PREFIX_READY

if not CAN_RUN:
    result = {'ok': False, 'skipped': True, 'RUN_EVOLUTION': RUN_EVOLUTION, 'CHECKS_OK': CHECKS_OK, 'DRY_RUN_OK': DRY_RUN_OK, 'REDIS_PREFIX_READY': REDIS_PREFIX_READY}
    display(result)
else:
    runner = evo.start_gigaevolve_background if RUN_IN_BACKGROUND else evo.run_gigaevolve
    result = runner(
        hash_id=HASH_ID,
        dataset=DATASET,
        run_name=RUN_NAME,
        max_pairs=MAX_PAIRS,
        budget=BUDGET,
        generations=GENERATIONS,
        eval_backend=EVAL_BACKEND,
        eval_workers=EVAL_WORKERS,
        worker_cap=WORKER_CAP,
        redis_host=REDIS_HOST,
        redis_port=REDIS_PORT,
        redis_db=REDIS_DB,
        llm_source=LLM_SOURCE,
        model=MODEL,
        local_base_url=LOCAL_BASE_URL,
        max_mutations_per_generation=MAX_MUTATIONS_PER_GENERATION,
        max_elites_per_generation=MAX_ELITES_PER_GENERATION,
    stage_timeout=STAGE_TIMEOUT,
    dag_timeout=DAG_TIMEOUT,
        llm_temperature=LLM_TEMPERATURE,
        llm_max_retries=LLM_MAX_RETRIES,
        llm_max_tokens=LLM_MAX_TOKENS,
        llm_request_timeout=LLM_REQUEST_TIMEOUT,
        resume=RESUME_RUN,
        clear_existing_prefix=CLEAR_REDIS_PREFIX_BEFORE_RUN,
        problem_output_dir=PROBLEM_OUTPUT_DIR,
        seed_config=SEED_CONFIG,
        seed_attacks=SEED_ATTACKS,
    require_proxy=REQUIRE_PROXY,
        log_dir=LOG_DIR,
        dry_run=False,
    )
    display(result)

if result.get('run_dir'):
    print('Viewer notebook can attach to:')
    print(result['run_dir'])


## Monitor

`tqdm` here tracks completed evolution generations, matching the old GigaEvo UI progress bar.

In [ ]:
if result.get('run_dir') and RUN_IN_BACKGROUND and MONITOR_IN_THIS_NOTEBOOK:
    monitor = evo.monitor_gigaevolve_run(
        result['run_dir'],
        poll_sec=2.0,
        show_progress=True,
        progress_mode='generations',
        total_generations=GENERATIONS,
    )
    display(monitor)
else:
    print('Monitor skipped. Use notebooks/43_gigaevolve_live_log_viewer.ipynb for live logs.')

## Metrics

In [ ]:
metrics_path = result.get('metrics_path') or (str(Path(result['run_dir']) / 'metrics.jsonl') if result.get('run_dir') else None)
if metrics_path:
    metrics_table = evo.read_metrics_jsonl(metrics_path)
    if hasattr(metrics_table, 'empty') and not metrics_table.empty:
        cols = [c for c in ['event_index', 'time', 'program_id', 'score', 'best_score_so_far', 'target_metric', 'asr', 'mean_l2_raw', 'mean_queries'] if c in metrics_table.columns]
        display(metrics_table[cols].tail(20))
    else:
        display(metrics_table)
    print('Metrics:', metrics_path)
    if result.get('log_path'):
        print('Log:', result.get('log_path'))

## Export / Snapshot

In [ ]:
# Exports the best programs from Redis; also works for a run started earlier (same RUN_NAME).
ready_to_export = EXPORT_AFTER_RUN

if ready_to_export:
    export_result = evo.export_best_program(
        prefix=RUN_NAME,
        redis_host=REDIS_HOST,
        redis_port=REDIS_PORT,
        redis_db=REDIS_DB,
        output_dir=EXPORT_DIR,
    )
    display(export_result)

    snapshot = evo.save_run_snapshot(
        prefix=RUN_NAME,
        file=Path(SNAPSHOT_DIR) / f'{RUN_NAME}.redis_snapshot.json',
        redis_host=REDIS_HOST,
        redis_port=REDIS_PORT,
        redis_db=REDIS_DB,
    )
    display(snapshot)
else:
    print('Export skipped (EXPORT_AFTER_RUN=False).')

In [ ]:
print(RUN_NAME)